## SECTION 1: Identify URL's to scrape

First identify the url's that lead to the relevant responses

In [7]:
# Node 1: Get all the base URL's to scrape


# Libraries
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.action_chains import ActionChains
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Setup options for Edge (you can change this if needed)
edge_options = Options()

# Initialize the webdriver for Microsoft Edge
driver = webdriver.Edge(options=edge_options)

# Specify the URL of the website you want to scrape
url = 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/list'

# Open the webpage
driver.get(url)

# Wait for a few seconds to let the page load (you may adjust this time or use WebDriverWait)
time.sleep(3)



# Create a list to store URLs and the associated text
data = []
scraped_urls = set()  # Track URLs we've already scraped

# Define a function to scrape the URLs and handle pagination
def scrape_urls():
    unique_id = 1  # Initialize a unique ID counter
    while True:
        # Keep track of the number of new URLs added
        current_urls = set()  # Set to store URLs from this page

        try:
            # Locate the section containing the URLs using a CSS selector (update this to match your HTML structure)
            container = driver.find_element(By.CSS_SELECTOR, '.order-2.sm\\:order-1.flex-auto.relative.sm\\:w-3\\/4')

            # Find all <a> elements within the container
            links = container.find_elements(By.TAG_NAME, 'a')

            # Extract the href attribute (URLs) and the text of each <a> tag
            for link in links:
                url = link.get_attribute('href')
                text = link.text.strip()  # Strip any extra spaces/newlines
                if url and url not in scraped_urls:
                    scraped_urls.add(url)  # Add the URL to the set
                    data.append({"ID": unique_id, "URL_MAIN": url, "Owner": text})
                    current_urls.add(url)  # Track new URLs
                    unique_id += 1  # Increment the unique ID

            # If no new URLs were added, it means we've hit the last page or the page was refreshed
            if not current_urls:
                print("No new URLs found. Ending loop.")
                break  # Exit the loop if no new URLs were added

            # Locate the "Next" button by its aria-label attribute
            next_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[aria-label="Next"]'))
            )

            # Check if the "Next" button is enabled (i.e., we haven't reached the last page)
            if "disabled" in next_button.get_attribute("class"):
                print("Last page reached.")
                break  # Exit the loop if the button is disabled (i.e., on the last page)

            # Scroll to the next button to make sure it's in view (this helps if the button is off-screen)
            actions = ActionChains(driver)
            actions.move_to_element(next_button).perform()

            # Click the "Next" button to go to the next page
            next_button.click()

            # Wait for the next page to load
            time.sleep(3)  # Adjust the wait time as needed for the page to load

        except Exception as e:
            # If an error occurs (e.g., the "Next" button is not found), exit the loop
            print(f"Error or reached last page: {e}")
            break

# Start scraping
scrape_urls()

# Create a DataFrame from the data collected
df = pd.DataFrame(data)

# Display the DataFrame
print(df)

# Optionally, you can save the DataFrame to a CSV file
df.to_csv('../data/raw/raw_submission_list.csv', index=False)

# Close the driver after scraping all pages
driver.quit()

No new URLs found. Ending loop.
      ID                                           URL_MAIN  \
0      1  https://consult.industry.gov.au/ai-mandatory-g...   
1      2  https://consult.industry.gov.au/ai-mandatory-g...   
2      3  https://consult.industry.gov.au/ai-mandatory-g...   
3      4  https://consult.industry.gov.au/ai-mandatory-g...   
4      5  https://consult.industry.gov.au/ai-mandatory-g...   
..   ...                                                ...   
274  275  https://consult.industry.gov.au/ai-mandatory-g...   
275  276  https://consult.industry.gov.au/ai-mandatory-g...   
276  277  https://consult.industry.gov.au/ai-mandatory-g...   
277  278  https://consult.industry.gov.au/ai-mandatory-g...   
278  279  https://consult.industry.gov.au/ai-mandatory-g...   

                                               Owner  
0                                      CPA Australia  
1                     Commonwealth Bank of Australia  
2                               Digital Rights

# SECTION 2:Response Scraping

For each response url identifed, visit each site and scrape the response

In [ ]:
# pip install Spire.Doc

In [89]:
# Node 2: Visit each link, and extract the text

import json
import re
import csv   
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
import time
import os
import glob
import PyPDF2
from docx import Document
from pathlib import Path
import shutil
from spire.doc import *
from spire.doc.common import *

# Write Document processing fuctions 
def get_latest_file(download_dir):
    # Get all files in the download directory
    files = glob.glob(os.path.join(download_dir, '*'))
    # Define the processed folder path
    processed_folder = os.path.join(download_dir, 'Processed')

    # Filter out any directories (including 'Processed') from the list
    files = [file for file in files if os.path.isfile(file) and not file.startswith(processed_folder)]
    
    if not files:
        return None

    # Return the most recently modified file
    latest_file = max(files, key=os.path.getmtime)
    return latest_file

def extract_text_from_pdf(pdf_path):
    text = ''
    with open(pdf_path, 'rb') as file:
        # Extract text from each page
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text()
    return text

def extract_text_from_docx(docx_path):
    
    # Create a Document object
    document = Document()
    # Load a Word document
    document.LoadFromFile(docx_path)

    # Extract the text of the document
    document_text = document.GetText()

    return document_text

def extract_text_from_pptx(file_path):
    presentation = Presentation(file_path)
    total_text = ''

    for slide in presentation.slides:
        title = ""
        text = ""

        # Extract title
        if slide.shapes.title and slide.shapes.title.has_text_frame:
            title = slide.shapes.title.text

        # Extract text from other shapes
        for shape in slide.shapes:
            if shape.has_text_frame and shape != slide.shapes.title:  # Exclude the title shape
                text += shape.text + "\n"  # Add a newline for separation

        # Combine title and text into a single string
        combined_text = f"{title}\n{text.strip()}\n-\n-\n-\n"  # Add seperator to mark a new slide
        total_text += combined_text

    return total_text


def move_to_processed(download_dir, filename):
    # Define the processed folder path
    processed_folder = os.path.join(download_dir, 'Processed')
    
    # Make sure the 'Processed' folder exists
    if not os.path.exists(processed_folder):
        os.makedirs(processed_folder)

    # Define the path for the new file location
    new_location = os.path.join(processed_folder, os.path.basename(filename))
    
    # Move the file to the 'Processed' folder
    shutil.move(filename, new_location)
    
    # Return the new location as a string
    return new_location


def wait_for_downloads(download_dir, timeout=60):
    """Wait for downloads to finish in the specified directory."""
    start_time = time.time()
    
    while True:
        # Get all files in the download directory
        files = glob.glob(os.path.join(download_dir, '*'))
        # Define the processed folder path
        processed_folder = os.path.join(download_dir, 'Processed')

        # Filter out any directories (including 'Processed') from the list
        files = [file for file in files if os.path.isfile(file) and not file.startswith(processed_folder)]
        print(f"Files in download directory: {files}")

        # If there are no files, wait a bit and check again
        if not files:
            time.sleep(3)
            continue
        
        # Check if any files are still being downloaded (files with .part extension)
        downloading_files = [file for file in files if file.endswith('.tmp')]
        
        # If no .part files, it means all downloads have completed
        if not downloading_files:
            print("All downloads completed.")
            break
        
        # If there are still downloading files, check the timeout condition
        if time.time() - start_time > timeout:
            print("Timeout waiting for downloads to complete.")
            break
        
        # Wait for a few seconds before checking again
        time.sleep(3)

def process_PDF_Link(pdf_url):
    download_dir = r'C:\Users\james.barro\Documents\rsm-nvidia-hackathon\data\Submission_Docs' 
    # Ensure the download directory exists
    if not os.path.exists(download_dir):
        os.makedirs(download_dir)

    # Get the current working directory
    current_directory = os.getcwd()

    # Set up WebDriver options
    options = Options()
    options.add_experimental_option('prefs', {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False, #To auto download the file
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
    "plugins.always_open_pdf_externally": True, #It will not show PDF directly in chrome
    "download.extensions_to_open": "",  # Disable auto-opening of files
    "profile.default_content_settings.popups": 0  # Disable popups

    })
    edge_driver_path = "C:/Users/james.barro//Documents/Generative AI - Hackathon/Webdrivers/edgedriver_win64/msedgedriver.exe"

    # Initialize WebDriver with Edge
    driver = webdriver.Edge(service=Service(edge_driver_path), options=options)

    # Settings
    driver.get('edge://settings/downloads')
    toggle = driver.execute_script('''
            return document.querySelector(' input[aria-label="Open Office files in the browser"]');
    ''')
    toggle.click()
    
    try:
        # Open a blank page
        driver.get("data:text/html,<html></html>")

        # Use JavaScript to create a download link and click it
        script = f"""
        var link = document.createElement('a');
        link.href = "{pdf_url}";
        link.download = 'Melbourne_Office_EOM_October_2023.pdf';
        link.style.display = 'none'; // Hide the link
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
        """
        driver.execute_script(script)

        # Wait for download to complete
        wait_for_downloads(download_dir)
        print("Download should have started.")

    finally:
        driver.quit()

    
     # Find the most recently downloaded file
    latest_file = get_latest_file(download_dir)

    if latest_file:
        print(f"Processing file: {latest_file}")

        # Check the file extension and extract text accordingly
        if latest_file.lower().endswith('.pdf'):
            text = extract_text_from_pdf(latest_file)
            docType = 'pdf'
        elif latest_file.lower().endswith('.docx') or latest_file.lower().endswith('.doc'):
            text = extract_text_from_docx(latest_file)
            docType = 'docx'
        elif latest_file.lower().endswith('.pptx'):
            text = extract_text_from_pptx(latest_file)
            docType = 'pptx'
        else:
            text = ''
            docType = 'NULL'
            print("Unsupported file format.")
            return text, docType
        
        # Get document Title
        file_name = os.path.basename(latest_file)
            # Remove the file extension
        filename_without_extension, _ = os.path.splitext(file_name)
        
        # Move it
        directory = move_to_processed(download_dir, latest_file)
        
#         # Don't Delete the file (Might plug into NVIDIA extractors) [Moved into Processed]
#         os.remove(latest_file)
#         print(f"File {latest_file} has been deleted.")
    else:
        print("No file found for processing.")
        return
    
    return (text, docType, filename_without_extension, file_name, directory)

###################################################################


def main(run_mode = 'resume'):
    # Setup options for Edge (you can change this if needed)
    edge_options = Options()

    # Initialize the webdriver for Microsoft Edge
    driver = webdriver.Edge(options=edge_options)

    # Read the CSV file containing URLs (the output file from the previous script)
    df = pd.read_csv('../data/raw/raw_submission_list.csv')

    # Path to track processed URLs
    processed_urls_file = '../data/raw/processed_url_list.txt'
    # Create or open a CSV file to store the results (open in append mode if it already exists)
    file_path = '../data/raw/raw_scraped_dataset.json'

    # Check if the file already exists, and skip writing header if it does
    file_exists = os.path.isfile(file_path)
    
    # If rerun is selected, clear the processed URLs file
    if run_mode == 'rerun':
        if os.path.exists(processed_urls_file):
            os.remove(processed_urls_file)
        if os.path.exists(file_path):
            os.remove(file_path)
        

    # Read existing JSON file if it exists, else initialize an empty list
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as json_file:
            try:
                existing_data = json.load(json_file)
            except json.JSONDecodeError:
                existing_data = []  # In case the file is empty or corrupted
    else:
        existing_data = []
            
    # Load previously processed URLs (if the file exists and we are resuming)
    if run_mode == 'resume' and os.path.exists(processed_urls_file):
        with open(processed_urls_file, 'r') as f:
            processed_urls = set(line.strip() for line in f.readlines())
    else:
        processed_urls = set()

    # Open the CSV in append mode, but avoid writing a blank line at the start
    with open(file_path, 'a', encoding='utf-8') as f:
        for index, row in df.iterrows():
            url = row['URL_MAIN']
            owner = row['Owner']

            # Skip URLs that have already been processed (only for resume mode)
            if run_mode == 'resume' and url in processed_urls:
                print(f"Skipping already processed URL: {url}")
                continue

            print(f"Visiting URL: {url}")

            # Open the URL
            driver.get(url)

            # Wait for the page to load
            time.sleep(3)

            # Now Check if there is any text to scrape from the webpage
            webpage = driver.find_element(By.TAG_NAME, "body")
            text = webpage.text
            # Remove the heading text
            text_sub_1 = ''.join(text.split('Share')[1:])
            # Remove the footer text
            text_sub_final = ''.join(text_sub_1.split('Privacy\nTerms of use\nAccessibility'))
                
            try:
                # Find all <a> elements within the container
                links = driver.find_elements(By.TAG_NAME, 'a')

                # Find the link related to the uploaded document
                to_visit = []
                for link in links:
                    url_total = link.get_attribute('href')
                    if 'download' in url_total:
                        to_visit.append(url_total)
                        print(url_total)

            except Exception as e:
                print(f"Error scraping content from {url}: {e}")
                continue  # Skip to the next URL if an error occurs

            # Counter to keep track of each document for the current URL
            source_index = 1
            
            # List to store the scraped data for this specific URL
            scraped_data_for_this_url = []
            
            # Scrape the documents linked to the current URL
            for i in to_visit:
                scraped = process_PDF_Link(i)  # Assume this function returns a tuple of scraped data

                # Prepare the row data to include the Source_Index
                row_data = {
                    'Owner': owner,
                    'URL_MAIN': url,
                    'Raw_Scraped_Text': scraped[0],
                    'Document_Name': scraped[2],
                    'File_Name': scraped[3],
                    'Document_Type': scraped[1],
                    'Directory': scraped[4],
                    'Source_Index': source_index
                }
                print(row_data)
                
                # Add the row data to the list for this URL
                scraped_data_for_this_url.append(row_data)
                
                # Open file in append mode and loop through data
                with open(file_path, mode='a', newline='', encoding='utf-8') as file:
                    writer = csv.writer(file)
                    writer.writerow(row_data)
#                 print(f'\n\nScraped Text: {scraped[0]}\n\n')

                # Increment the source_index for the next document
                source_index += 1
            
            # Now If there were no documents scraped, save the text from the webpage
            if source_index == 1:
                # Write Webpage to TXT file
                cleaned_owner = re.sub(r'\s+', '/', owner)
                txt_filename = 'Webpage_Submission_' + owner + '.txt'
                # Remove all white space
                cleaned_text_filename = re.sub(r'\s+', '', txt_filename)
                txt_filelocation = '../data/Submission_Docs/Processed/' + cleaned_text_filename
                row_data = {
                    'Owner': owner,
                    'URL_MAIN': url,
                    'Raw_Scraped_Text': text_sub_final,
                    'Document_Name': 'Website Submission',
                    'File_Name': cleaned_text_filename,
                    'Document_Type': 'txt',
                    'Directory': txt_filelocation,
                    'Source_Index': source_index
                }
                print(row_data)
                # Write the text to a .txt file
                with open(txt_filelocation, 'w', encoding='utf-8') as file:
                    file.write(text_sub_final)
                    print('\nExtracted Website Response\n')
                
                # Add the row data to the list for this URL
                scraped_data_for_this_url.append(row_data)
            
            # After processing this URL, add it to the processed URLs set
            processed_urls.add(url)

            # Save the list of processed URLs to the file to track progress
            with open(processed_urls_file, 'a', encoding='utf-8') as f_processed:
                f_processed.write(url + '\n')
                
                

            # Append the scraped data for this URL to the existing data
            existing_data.append(scraped_data_for_this_url)

            # Write the updated data to the JSON file
            with open(file_path, 'w', encoding='utf-8') as json_file:
                json.dump(existing_data, json_file, ensure_ascii=False, indent=4)

            # Clear the list to store data for the next URL
            scraped_data_for_this_url = []
            
    # Close the driver after scraping all URLs
    driver.quit()

    print("JSON updated with scraped text.")
    
    
####### RUN MAIN

# Mode toggle: choose 'resume' to resume processing or 'rerun' for a full rerun
run_mode = 'resume'  # Change to 'rerun' for a fresh run
main(run_mode)

Skipping already processed URL: https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/324
Skipping already processed URL: https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/321
Skipping already processed URL: https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/320
Skipping already processed URL: https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/319
Skipping already processed URL: https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/318
Skipping already processed URL: https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/317
Skipping already processed URL: https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/315
Skipping already processed URL: https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/314
Skipping already processed URL: https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/313
Skipping already processed U

https://consult.industry.gov.au/ai-mandatory-guardrails/download/fil30ddefa2b3101ab3df99f
Files in download directory: []
Files in download directory: ['C:\\Users\\james.barro\\Documents\\rsm-nvidia-hackathon\\data\\Submission_Docs\\Blackham_Submission_AI_guardrails.c6f8c14e.pdf']
All downloads completed.
Download should have started.
Processing file: C:\Users\james.barro\Documents\rsm-nvidia-hackathon\data\Submission_Docs\Blackham_Submission_AI_guardrails.c6f8c14e.pdf
{'Owner': 'A/Prof Alysia Blackham', 'URL_MAIN': 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/54', 'Raw_Scraped_Text': ' 1 Submission to the Safe and responsible AI in Australia: Proposals paper for introducing mandatory guardrails for AI in high-risk settings  A/Prof Alysia Blackham, Melbourne Law School, University of Melbourne  I write this submission as an expert in equality law, who has written on the discrimination risks of AI technologies and algorithmic discrimination.1  I commend the p

{'Owner': 'Pathology Technology Australia', 'URL_MAIN': 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/52', 'Raw_Scraped_Text': "\nPathology Technology Australia\nPublished name\nPathology Technology Australia\n1. Do the proposed principles adequately capture high-risk AI?\nYes\nPlease provide any additional comments.\nAlthough there may need to be further consideration to unintended consequences that capture medical devices that are exempt or excluded from the TGA's ARTG. If the TGA assesses such devices as such a low risk that inclusion on the ARTG is unnecessary, will the devices still need to comply with the 10 guardrails? We could help construct use cases that better differentiate low risk medical devices that should not be burdened by the guardrails. It is likely to be an unacceptable burden for industry to comply with a separate conformity assessment process if a medical device is exempt or excluded by the TGA (low risk) but requires DISR conformity ass

https://consult.industry.gov.au/ai-mandatory-guardrails/download/fil30dc98291a1c2733aeea1
Files in download directory: []
Files in download directory: ['C:\\Users\\james.barro\\Documents\\rsm-nvidia-hackathon\\data\\Submission_Docs\\wp_workforce_skills_forecast.6ad6ce1c.pdf']
All downloads completed.
Download should have started.
Processing file: C:\Users\james.barro\Documents\rsm-nvidia-hackathon\data\Submission_Docs\wp_workforce_skills_forecast.6ad6ce1c.pdf
{'Owner': 'Anonymous', 'URL_MAIN': 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/51', 'Raw_Scraped_Text': "AI Impact:  \n2024 Workforce  \nSkills Forecast\nHow emerging technologies will impact the  \nwork—and workforce—of the future\nANNUAL RESEARCH REPORT2\nCONTENTS03Introduction18Workforce skills trends\n04Methodology19United States\n06Top takeaway No. 127Canada\n09Top takeaway No. 235United Kingdom\n10Top takeaway No. 343Germany\n17New world, new workforce51India\n59Australia\n67Singapore\n75JapanAI 

{'Owner': 'Sheng Lan', 'URL_MAIN': 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/50', 'Raw_Scraped_Text': '\nSheng Lan\nPublished name\nSheng Lan\n1. Do the proposed principles adequately capture high-risk AI?\nYes\nAre there any principles we should add or remove?\nNo\n2. Do you have any suggestions for how the principles could better capture harms to First Nations people, communities and Country?\nNo\n3. Do the proposed principles, supported by examples, give enough clarity and certainty on high-risk AI settings and high-risk AI models? Is a more defined approach, with a list of illustrative uses, needed?\nYES the principles give enough clarity and certainty\n4. Are there high-risk use cases that government should consider banning in its regulatory response (for example, where there is an unacceptable level of risk)?\nYes\n5. Are the proposed principles flexible enough to capture new and emerging forms of high-risk AI, such as general-purpose AI (GPAI)?\nYe

{'Owner': 'Samuel Grew', 'URL_MAIN': 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/46', 'Raw_Scraped_Text': "\nSamuel Grew\nPublished name\nSamuel Grew\n6. Should mandatory guardrails apply to all GPAI models?\nNo\nPlease provide any additional comments.\nIn my view, the imposition of mandatory guardrails should not be uniformly applied to all Global Public AI (GPAI) models. Instead, these guardrails should be proportionate to the risks posed by the AI system. This approach aligns with the EU AI Act, which differentiates between GPAI and GPAI with systemic risk, thereby applying the strictest mandatory guardrails to those posing systemic risk.\nA risk-based approach to AI regulation is paramount. The principle-f factor, though effective in classifying high-risk narrow-AI systems based on the severity and extent of adverse impacts, should also be considered in defining GPAI and imposing specific guardrails.\nThe Government and developers should strive to build

... (24 outputs left out)

https://consult.industry.gov.au/ai-mandatory-guardrails/download/fil3124aaa8c65c70465617d
Files in download directory: []
Files in download directory: ['C:\\Users\\james.barro\\Documents\\rsm-nvidia-hackathon\\data\\Submission_Docs\\Enhanced_Submission_Document.ef0f5da6.c24d84dc.pdf']
All downloads completed.
Download should have started.
Processing file: C:\Users\james.barro\Documents\rsm-nvidia-hackathon\data\Submission_Docs\Enhanced_Submission_Document.ef0f5da6.c24d84dc.pdf
{'Owner': 'Ashish Khanna', 'URL_MAIN': 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/4', 'Raw_Scraped_Text': 'Enhanced  Submission  Document:  Strengthening  AI Regulatory  Framework \nBy: Ashish Khanna  \nEmail:  Ashishkhanna1985@gmail.com   \n \n1. Introduction  \n \nArtificial  Intelligence  (AI) is a pivotal  technology  shaping  the future  across  various \nsectors. The Australian Government’s initiative to regulate high -risk AI marks a \nsignificant step  toward  ensuring  the s

{'Owner': 'Knowz', 'URL_MAIN': 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/3', 'Raw_Scraped_Text': '\nKnowz\nPublished name\nKnowz\n1. Do the proposed principles adequately capture high-risk AI?\nNo\nPlease provide any additional comments.\nIn response to whether the proposed principles adequately capture high-risk AI, it appears that the principles outlined in the document are comprehensive and relevant to addressing the potential risks that AI poses in various contexts, particularly in areas such as human rights, health, safety, legal effects, and systemic impacts. The focus on severity and extent of adverse impacts provides a robust framework for ensuring that AI systems operate ethically, safely, and transparently.\nHowever, there is always room for further refinement, particularly in emphasizing continuous oversight by developers and deployers, especially as AI systems evolve beyond their initial design. The responsibility to maintain oversight is crit

https://consult.industry.gov.au/ai-mandatory-guardrails/download/fil305581091d43d7702070d
Files in download directory: []
Files in download directory: ['C:\\Users\\james.barro\\Documents\\rsm-nvidia-hackathon\\data\\Submission_Docs\\Butler_and_Lupton_2024_Generative_AI_Technologies_Applied_to_Ecosystems_and_the_Environment_working_paper.f7a6d59f.pdf']
All downloads completed.
Download should have started.
Processing file: C:\Users\james.barro\Documents\rsm-nvidia-hackathon\data\Submission_Docs\Butler_and_Lupton_2024_Generative_AI_Technologies_Applied_to_Ecosystems_and_the_Environment_working_paper.f7a6d59f.pdf
{'Owner': 'Professor Deborah Lupton', 'URL_MAIN': 'https://consult.industry.gov.au/ai-mandatory-guardrails/submission/view/1', 'Raw_Scraped_Text': 'PROJECT REPORT  \nadmscentre.org.au/publications  \nADM+S  WORKING \nPAPER SERIES  \nGenerative AI Technologies Applied to \nEcosystems and the Natural Environment  \nA Scoping Review  \nMay 2024  \nAuthors:  \nElla Butler \nDeborah L

CSV updated with scraped text.
